# OLD

In [1]:
import vitaldb
import random
import numpy as np
import pandas as pd
import scipy.signal
import matplotlib.pyplot as plt

# Constants
SRATE = 128  # Hz
SEGLEN = 10 * SRATE  # 10 seconds in samples
MAX_CASES = 200

# Data Loading
df_trks = pd.read_csv("https://api.vitaldb.net/trks")
df_cases = pd.read_csv("https://api.vitaldb.net/cases")

# Track selection
track_names = ['BIS/EEG1_WAV', 'BIS/BIS']
EEG = 0
BIS = 1

# Filter based on inclusion/exclusion criteria
caseids = set(df_cases.loc[df_cases['age'] > 18, 'caseid'])
#caseids &= set(df_trks.loc[df_trks['tname'] == 'BIS/EEG2_WAV', 'caseid'])
caseids &= set(df_trks.loc[df_trks['tname'] == 'BIS/EEG1_WAV', 'caseid'])
caseids &= set(df_trks.loc[df_trks['tname'] == 'BIS/BIS', 'caseid'])
caseids -= set(df_trks.loc[df_trks['tname'] == 'Primus/EXP_DES', 'caseid'])
caseids -= set(df_trks.loc[df_trks['tname'] == 'Orchestra/PPF20_CE', 'caseid'])
caseids -= set(df_trks.loc[df_trks['tname'] == 'Orchestra/RFTN50_CE', 'caseid'])
caseids = list(caseids)

# Data extraction initialization
x = []
y = []
c = []
icase = 0

# **Updated**: Bandpass filter setup (one-time initialization)
# This avoids repeatedly initializing the filter in every iteration of the loop
lowcut = 0.5
highcut = 50.0
order = 4
b, a = scipy.signal.butter(order, [lowcut, highcut], btype='band', fs=SRATE)

# Processing each case
for caseid in caseids:
    print(f'Loading caseid={caseid} ({icase + 1}/{MAX_CASES})...', end='')

    # Load case data
    vals = vitaldb.load_case(caseid, track_names, 1 / SRATE)
    op_st = df_cases[df_cases['caseid'] == caseid]['opstart'].values[0]
    op_end = df_cases[df_cases['caseid'] == caseid]['opend'].values[0]

    # Exclusion Criteria: **Updated** logging to inform about skipped cases
    if not np.any(vals[:, BIS] > 0):
        print('Skipping due to all BIS <= 0')
        continue
    if len(vals) < 1800 * SRATE:
        print('Skipping due to short length')
        continue

    # Fill missing BIS values with forward fill: **Kept as is**
    vals[:, BIS:] = pd.DataFrame(vals[:, BIS:]).ffill(limit=7*SRATE).values

    # Extract and filter segments
    for isamp in range(SEGLEN, len(vals), 2 * SRATE):
        if op_st <= isamp / SRATE <= op_end:
            bis = vals[isamp, BIS]
            if np.isnan(bis) or bis == 0:
                continue
            eeg = vals[isamp - SEGLEN:isamp, EEG]
            eeg_filtered = scipy.signal.filtfilt(b, a, eeg)  # Apply filter
            # **Updated**: Normalize EEG signal to zero mean and unit variance
            eeg_filtered = (eeg_filtered - np.mean(eeg_filtered)) / np.std(eeg_filtered)  # Normalization
            x.append(eeg_filtered)
            y.append(bis)
            c.append(caseid)

    # Print progress: **Kept as is**
    icase += 1
    print(f'{len(y)} segments loaded so far')

    if icase >= MAX_CASES:
        break

# Convert to numpy arrays: **Kept as is**
x = np.array(x)
y = np.array(y)
c = np.array(c)


import joblib

# Assuming x, y, c are already populated as NumPy arrays
data = {'x': x, 'y': y, 'c': c}

# Save the data dictionary to a file
joblib.dump(data, 'data_10_sec_200.pkl')


Loading caseid=2 (1/200)...6450 segments loaded so far
Loading caseid=4 (2/200)...14081 segments loaded so far
Loading caseid=10 (3/200)...22761 segments loaded so far
Loading caseid=12 (4/200)...35306 segments loaded so far
Loading caseid=18 (5/200)...Skipping due to all BIS <= 0
Loading caseid=21 (5/200)...39504 segments loaded so far
Loading caseid=24 (6/200)...41004 segments loaded so far
Loading caseid=25 (7/200)...46990 segments loaded so far
Loading caseid=27 (8/200)...53797 segments loaded so far
Loading caseid=33 (9/200)...55252 segments loaded so far
Loading caseid=39 (10/200)...Skipping due to all BIS <= 0
Loading caseid=43 (10/200)...60763 segments loaded so far
Loading caseid=49 (11/200)...63926 segments loaded so far
Loading caseid=56 (12/200)...76068 segments loaded so far
Loading caseid=58 (13/200)...81465 segments loaded so far
Loading caseid=61 (14/200)...84527 segments loaded so far
Loading caseid=62 (15/200)...87662 segments loaded so far
Loading caseid=64 (16/200).

['data_10_sec_200.pkl']

We constructed three cohorts from VitalDB under a common eligibility: adults (age ≥ 18) with concurrent BIS/BIS and BIS/EEG1_WAV recordings. Modality tags were inferred from device tracks as proxies: cases with Orchestra/PPF20_CE or Orchestra/RFTN50_CE were tagged “TCI,” cases with volatile exp tracks (e.g., Primus/EXP_DES) were tagged “volatile,” and cases with both or neither were tagged “mixed” or “unknown,” respectively. We defined: (A) modality-agnostic (all eligible cases), (B) volatile-only (volatile present, TCI absent), and (C) TCI-only (TCI present, volatile absent). Track presence was used only for cohorting/stratification, not as model inputs. All preprocessing, windowing, and labeling parameters were held constant across cohorts. We report performance overall and stratified by modality and evaluate cross-cohort generalization (Train-A→Eval-B/C; Train-B→Eval-C; Train-C→Eval-B).

| Dimension                | **A) Modality-Agnostic** | **B) Volatile-Only**         | **C) TCI-Only**                  |
| ------------------------ | ------------------------ | ---------------------------- | -------------------------------- |
| Dataset size             | **Largest**              | Medium–Small                 | **Smallest** (usually)           |
| External validity        | **Highest**              | Limited to volatile rooms    | Limited to TCI rooms             |
| Cohort homogeneity       | Low–Medium               | **High**                     | **High**                         |
| Optimization difficulty  | Medium                   | **Low**                      | **Low**                          |
| Overfitting risk         | Lower (more data)        | Medium (less data)           | **Higher** (smallest data)       |
| Domain-shift sensitivity | N/A                      | **High** when tested on TCI  | **High** when tested on volatile |
| Best use case            | General-purpose model    | Volatile-centric deployments | TCI-centric deployments          |


Recommended Experiment Plan (for the paper)

Within-cohort models

Train A → Eval A (primary)

Train B → Eval B

Train C → Eval C

Cross-cohort generalization

Train A → Eval B and Eval C (robustness of general model)

Train B → Eval C (volatile → TCI transfer)

Train C → Eval B (TCI → volatile transfer)

Report (at minimum)

MAE/RMSE/CCC (mean ± SD across patient-wise folds)

Calibration (ECE + reliability curves)

Per-modality slices (for A)

Sample sizes: #cases, #segments, median segments per case

Brief statistical comparison (paired test) for A→B vs B→B, and A→C vs C→C

**Approach A — Modality-agnostic (general model).**
This cohort includes all adult cases that have the required BIS index and EEG waveform, regardless of how anesthesia was delivered. In practice, that means you keep volatile, TCI, mixed, and unknown-modality cases together and train a single model without feeding any drug/device tracks as inputs. The value of A is realism and scale: it mirrors the heterogeneity of real operating rooms and usually yields the largest sample size, which stabilizes training and improves statistical power. The trade-off is that the data are more diverse—EEG dynamics, EMG contamination, surgical phases, and recording habits vary across devices and sites—so optimization can be slightly harder and performance may differ across subgroups. Interpreting results from A works best when you also examine slices by modality (e.g., volatile vs TCI) to reveal domain gaps. If the goal is a broadly deployable DoA model, A is the natural baseline; you can fairly claim “general-purpose” performance while transparently reporting how it behaves across modalities.

**Approach B — Volatile-only (vaporizer workflows).**
This cohort keeps only cases where volatile agent tracks are present and removes any case with TCI tracks (and typically excludes mixed cases) so that the remaining EEG reflects inhalational anesthesia workflows. The benefit is a more homogeneous cohort: volatile agents often produce relatively stereotyped EEG patterns during maintenance (e.g., strong alpha with dose-dependent slowing), which can make learning easier and sometimes lift in-domain metrics. However, B sacrifices external validity and sample size: it won’t represent TCI rooms, and it can inherit site/device biases because inclusion depends on whether volatile tracks were logged. Unknown-modality cases are usually excluded to keep the cohort “pure,” but that further reduces data. Models trained under B typically perform well within volatile cases yet degrade when evaluated on TCI cohorts, quantifying a domain shift. The right way to present B is as a focused model for volatile settings, with clear scope and limitations.

**Approach C — TCI-only (propofol/remifentanil Ce-guided).**
This cohort retains cases with TCI effect-site tracks and removes volatile and mixed cases, isolating propofol/remifentanil workflows. Like B, C creates a homogeneous learning problem, but it is often the smallest cohort because TCI logging is less ubiquitous than basic monitors. EEG under propofol-dominant TCI can differ meaningfully from volatile anesthesia (e.g., stronger frontal alpha with characteristic beta/slow dynamics and different EMG profiles), so a C-trained model can capture TCI-specific structure and calibrate well in those rooms. The cost is narrow applicability: it usually generalizes poorly to volatile cases, and results may reflect device or site practices tied to Orchestra usage. When reporting C, position it as a TCI-focused model; emphasize in-domain performance, acknowledge reduced sample size (and higher variance across folds), and use cross-cohort tests to show how much performance drops when the model is applied outside TCI environments.


# Type-A

In [4]:
import vitaldb
import random
import numpy as np
import pandas as pd
import scipy.signal
import matplotlib.pyplot as plt

# Constants
SRATE = 128  # Hz
SEGLEN = 10 * SRATE  # 10 seconds in samples
MAX_CASES = 1000

# Data Loading
df_trks = pd.read_csv("https://api.vitaldb.net/trks")
df_cases = pd.read_csv("https://api.vitaldb.net/cases")

# Track selection
track_names = ['BIS/EEG1_WAV', 'BIS/BIS']
EEG = 0
BIS = 1

# A: Modality-agnostic
caseids = set(df_cases.loc[df_cases['age'] >= 18, 'caseid'])
caseids &= set(df_trks.loc[df_trks['tname'] == 'BIS/EEG1_WAV', 'caseid'])
caseids &= set(df_trks.loc[df_trks['tname'] == 'BIS/BIS', 'caseid'])
# DO NOT subtract Primus/Orchestra here
caseids = list(caseids)
random.seed(42); random.shuffle(caseids)


# Data extraction initialization
x = []
y = []
c = []
icase = 0

# **Updated**: Bandpass filter setup (one-time initialization)
# This avoids repeatedly initializing the filter in every iteration of the loop
lowcut = 0.5
highcut = 50.0
order = 4
b, a = scipy.signal.butter(order, [lowcut, highcut], btype='band', fs=SRATE)

# Processing each case
for caseid in caseids:
    print(f'Loading caseid={caseid} ({icase + 1}/{MAX_CASES})...', end='')

    # Load case data
    vals = vitaldb.load_case(caseid, track_names, 1 / SRATE)
    op_st = df_cases[df_cases['caseid'] == caseid]['opstart'].values[0]
    op_end = df_cases[df_cases['caseid'] == caseid]['opend'].values[0]

    # Exclusion Criteria: **Updated** logging to inform about skipped cases
    if not np.any(vals[:, BIS] > 0):
        print('Skipping due to all BIS <= 0')
        continue
    if len(vals) < 1800 * SRATE:
        print('Skipping due to short length')
        continue

    # Fill missing BIS values with forward fill: **Kept as is**
    vals[:, BIS:] = pd.DataFrame(vals[:, BIS:]).ffill(limit=7*SRATE).values

    # Extract and filter segments
    for isamp in range(SEGLEN, len(vals), 2 * SRATE):
        if op_st <= isamp / SRATE <= op_end:
            bis = vals[isamp, BIS]
            if np.isnan(bis) or bis == 0:
                continue
            eeg = vals[isamp - SEGLEN:isamp, EEG]
            eeg_filtered = scipy.signal.filtfilt(b, a, eeg)  # Apply filter
            # **Updated**: Normalize EEG signal to zero mean and unit variance
            eeg_filtered = (eeg_filtered - np.mean(eeg_filtered)) / np.std(eeg_filtered)  # Normalization
            x.append(eeg_filtered)
            y.append(bis)
            c.append(caseid)

    # Print progress: **Kept as is**
    icase += 1
    print(f'{len(y)} segments loaded so far')

    if icase >= MAX_CASES:
        break

# Convert to numpy arrays: **Kept as is**
x = np.array(x)
y = np.array(y)
c = np.array(c)


import joblib

# Assuming x, y, c are already populated as NumPy arrays
data = {'x': x, 'y': y, 'c': c}

# Save the data dictionary to a file
joblib.dump(data, 'A_data_10_sec_1000.pkl')


Loading caseid=6005 (1/1000)...2995 segments loaded so far
Loading caseid=5017 (2/1000)...5096 segments loaded so far
Loading caseid=4954 (3/1000)...7944 segments loaded so far
Loading caseid=1155 (4/1000)...13112 segments loaded so far
Loading caseid=4189 (5/1000)...18657 segments loaded so far
Loading caseid=663 (6/1000)...21914 segments loaded so far
Loading caseid=4792 (7/1000)...26715 segments loaded so far
Loading caseid=1022 (8/1000)...33760 segments loaded so far
Loading caseid=4524 (9/1000)...35111 segments loaded so far
Loading caseid=486 (10/1000)...42632 segments loaded so far
Loading caseid=3082 (11/1000)...48632 segments loaded so far
Loading caseid=2185 (12/1000)...58826 segments loaded so far
Loading caseid=153 (13/1000)...62112 segments loaded so far
Loading caseid=823 (14/1000)...64063 segments loaded so far
Loading caseid=5062 (15/1000)...65473 segments loaded so far
Loading caseid=5382 (16/1000)...67714 segments loaded so far
Loading caseid=5961 (17/1000)...70410 se

['A_data_10_sec_1000.pkl']

# Type-B

In [1]:
import vitaldb
import random
import numpy as np
import pandas as pd
import scipy.signal
import matplotlib.pyplot as plt

# Constants
SRATE = 128  # Hz
SEGLEN = 10 * SRATE  # 10 seconds in samples
MAX_CASES = 800

# Data Loading
df_trks = pd.read_csv("https://api.vitaldb.net/trks")
df_cases = pd.read_csv("https://api.vitaldb.net/cases")

# Track selection
track_names = ['BIS/EEG1_WAV', 'BIS/BIS']
EEG = 0
BIS = 1

# B: Volatile-only
caseids = set(df_cases.loc[df_cases['age'] >= 18, 'caseid'])
caseids &= set(df_trks.loc[df_trks['tname'] == 'BIS/EEG1_WAV', 'caseid'])
caseids &= set(df_trks.loc[df_trks['tname'] == 'BIS/BIS', 'caseid'])

# Markers
volatile = set(df_trks.loc[df_trks['tname'] == 'Primus/EXP_DES', 'caseid'])
tci = set(df_trks.loc[df_trks['tname'].isin(['Orchestra/PPF20_CE','Orchestra/RFTN50_CE']), 'caseid'])

# Keep volatile, drop TCI (also removes mixed)
caseids &= volatile
caseids -= tci

caseids = list(caseids)
random.seed(42); random.shuffle(caseids)

# Data extraction initialization
x = []
y = []
c = []
icase = 0

# **Updated**: Bandpass filter setup (one-time initialization)
# This avoids repeatedly initializing the filter in every iteration of the loop
lowcut = 0.5
highcut = 50.0
order = 4
b, a = scipy.signal.butter(order, [lowcut, highcut], btype='band', fs=SRATE)

# Processing each case
for caseid in caseids:
    print(f'Loading caseid={caseid} ({icase + 1}/{MAX_CASES})...', end='')

    # Load case data
    vals = vitaldb.load_case(caseid, track_names, 1 / SRATE)
    op_st = df_cases[df_cases['caseid'] == caseid]['opstart'].values[0]
    op_end = df_cases[df_cases['caseid'] == caseid]['opend'].values[0]

    # Exclusion Criteria: **Updated** logging to inform about skipped cases
    if not np.any(vals[:, BIS] > 0):
        print('Skipping due to all BIS <= 0')
        continue
    if len(vals) < 1800 * SRATE:
        print('Skipping due to short length')
        continue

    # Fill missing BIS values with forward fill: **Kept as is**
    vals[:, BIS:] = pd.DataFrame(vals[:, BIS:]).ffill(limit=7*SRATE).values

    # Extract and filter segments
    for isamp in range(SEGLEN, len(vals), 2 * SRATE):
        if op_st <= isamp / SRATE <= op_end:
            bis = vals[isamp, BIS]
            if np.isnan(bis) or bis == 0:
                continue
            eeg = vals[isamp - SEGLEN:isamp, EEG]
            eeg_filtered = scipy.signal.filtfilt(b, a, eeg)  # Apply filter
            # **Updated**: Normalize EEG signal to zero mean and unit variance
            eeg_filtered = (eeg_filtered - np.mean(eeg_filtered)) / np.std(eeg_filtered)  # Normalization
            x.append(eeg_filtered)
            y.append(bis)
            c.append(caseid)

    # Print progress: **Kept as is**
    icase += 1
    print(f'{len(y)} segments loaded so far')

    if icase >= MAX_CASES:
        break

# Convert to numpy arrays: **Kept as is**
x = np.array(x)
y = np.array(y)
c = np.array(c)


import joblib

# Assuming x, y, c are already populated as NumPy arrays
data = {'x': x, 'y': y, 'c': c}

# Save the data dictionary to a file
joblib.dump(data, 'B_data_10_sec_1000.pkl')


Loading caseid=1731 (1/800)...2817 segments loaded so far
Loading caseid=1932 (2/800)...14658 segments loaded so far
Loading caseid=6139 (3/800)...15538 segments loaded so far
Loading caseid=2502 (4/800)...21628 segments loaded so far
Loading caseid=4539 (5/800)...23566 segments loaded so far
Loading caseid=2323 (6/800)...27255 segments loaded so far
Loading caseid=3383 (7/800)...34534 segments loaded so far
Loading caseid=1231 (8/800)...47159 segments loaded so far
Loading caseid=825 (9/800)...49810 segments loaded so far
Loading caseid=2723 (10/800)...50441 segments loaded so far
Loading caseid=2005 (11/800)...Skipping due to all BIS <= 0
Loading caseid=5654 (11/800)...54931 segments loaded so far
Loading caseid=955 (12/800)...56731 segments loaded so far
Loading caseid=4916 (13/800)...63932 segments loaded so far
Loading caseid=5033 (14/800)...68132 segments loaded so far
Loading caseid=4409 (15/800)...74234 segments loaded so far
Loading caseid=6046 (16/800)...Skipping due to all B

['B_data_10_sec_1000.pkl']

# Type-C

In [1]:
import vitaldb
import random
import numpy as np
import pandas as pd
import scipy.signal
import matplotlib.pyplot as plt

# Constants
SRATE = 128  # Hz
SEGLEN = 10 * SRATE  # 10 seconds in samples
MAX_CASES = 1000

# Data Loading
df_trks = pd.read_csv("https://api.vitaldb.net/trks")
df_cases = pd.read_csv("https://api.vitaldb.net/cases")

# Track selection
track_names = ['BIS/EEG1_WAV', 'BIS/BIS']
EEG = 0
BIS = 1

# C: TCI-only
caseids = set(df_cases.loc[df_cases['age'] >= 18, 'caseid'])
caseids &= set(df_trks.loc[df_trks['tname'] == 'BIS/EEG1_WAV', 'caseid'])
caseids &= set(df_trks.loc[df_trks['tname'] == 'BIS/BIS', 'caseid'])

# Markers
volatile = set(df_trks.loc[df_trks['tname'] == 'Primus/EXP_DES', 'caseid'])
tci = set(df_trks.loc[df_trks['tname'].isin(['Orchestra/PPF20_CE','Orchestra/RFTN50_CE']), 'caseid'])

# Keep TCI, drop volatile (also removes mixed)
caseids &= tci
caseids -= volatile

caseids = list(caseids)
random.seed(42); random.shuffle(caseids)

# Data extraction initialization
x = []
y = []
c = []
icase = 0

# **Updated**: Bandpass filter setup (one-time initialization)
# This avoids repeatedly initializing the filter in every iteration of the loop
lowcut = 0.5
highcut = 50.0
order = 4
b, a = scipy.signal.butter(order, [lowcut, highcut], btype='band', fs=SRATE)

# Processing each case
for caseid in caseids:
    print(f'Loading caseid={caseid} ({icase + 1}/{MAX_CASES})...', end='')

    # Load case data
    vals = vitaldb.load_case(caseid, track_names, 1 / SRATE)
    op_st = df_cases[df_cases['caseid'] == caseid]['opstart'].values[0]
    op_end = df_cases[df_cases['caseid'] == caseid]['opend'].values[0]

    # Exclusion Criteria: **Updated** logging to inform about skipped cases
    if not np.any(vals[:, BIS] > 0):
        print('Skipping due to all BIS <= 0')
        continue
    if len(vals) < 1800 * SRATE:
        print('Skipping due to short length')
        continue

    # Fill missing BIS values with forward fill: **Kept as is**
    vals[:, BIS:] = pd.DataFrame(vals[:, BIS:]).ffill(limit=7*SRATE).values

    # Extract and filter segments
    for isamp in range(SEGLEN, len(vals), 2 * SRATE):
        if op_st <= isamp / SRATE <= op_end:
            bis = vals[isamp, BIS]
            if np.isnan(bis) or bis == 0:
                continue
            eeg = vals[isamp - SEGLEN:isamp, EEG]
            eeg_filtered = scipy.signal.filtfilt(b, a, eeg)  # Apply filter
            # **Updated**: Normalize EEG signal to zero mean and unit variance
            eeg_filtered = (eeg_filtered - np.mean(eeg_filtered)) / np.std(eeg_filtered)  # Normalization
            x.append(eeg_filtered)
            y.append(bis)
            c.append(caseid)

    # Print progress: **Kept as is**
    icase += 1
    print(f'{len(y)} segments loaded so far')

    if icase >= MAX_CASES:
        break

# Convert to numpy arrays: **Kept as is**
x = np.array(x)
y = np.array(y)
c = np.array(c)


import joblib

# Assuming x, y, c are already populated as NumPy arrays
data = {'x': x, 'y': y, 'c': c}

# Save the data dictionary to a file
joblib.dump(data, 'C_data_10_sec_1000.pkl')


Loading caseid=4620 (1/1000)...1932 segments loaded so far
Loading caseid=6329 (2/1000)...5619 segments loaded so far
Loading caseid=4514 (3/1000)...6935 segments loaded so far
Loading caseid=2872 (4/1000)...16092 segments loaded so far
Loading caseid=4177 (5/1000)...17205 segments loaded so far
Loading caseid=4234 (6/1000)...18847 segments loaded so far
Loading caseid=17 (7/1000)...26640 segments loaded so far
Loading caseid=140 (8/1000)...34583 segments loaded so far
Loading caseid=1078 (9/1000)...38605 segments loaded so far
Loading caseid=1551 (10/1000)...40221 segments loaded so far
Loading caseid=3264 (11/1000)...42302 segments loaded so far
Loading caseid=6298 (12/1000)...46904 segments loaded so far
Loading caseid=5 (13/1000)...55754 segments loaded so far
Loading caseid=2456 (14/1000)...58039 segments loaded so far
Loading caseid=3117 (15/1000)...66805 segments loaded so far
Loading caseid=1863 (16/1000)...74149 segments loaded so far
Loading caseid=1606 (17/1000)...76465 segm

['C_data_10_sec_1000.pkl']